# 02 - Descriptive Statistics and Outliers

This notebook studies distributional properties of the transformed financial series, with emphasis on non-normality and outliers.

In [1]:
from __future__ import annotations

import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import FIGURES_DIR, PROCESSED_DATA_DIR, RAW_DATA_DIR, RESULTS_DIR, SERIES_FILES
from src.utils import configure_logging, ensure_directory

configure_logging()
sns.set_theme(style="whitegrid")
ensure_directory(PROCESSED_DATA_DIR)
ensure_directory(RESULTS_DIR)
ensure_directory(FIGURES_DIR)

from src.outliers import detect_outliers_iqr, detect_outliers_zscore, outlier_summary
from src.statistics import correlation_matrix, descriptive_statistics, normality_tests
from src.visualization import plot_boxplots, plot_correlation_heatmap, plot_outlier_counts

## Load Processed Returns

Returns are used for distributional analysis because price levels are usually non-stationary and trend-dominated.

In [2]:
returns = pd.read_csv(PROCESSED_DATA_DIR / "market_data_returns.csv", parse_dates=["Date"], index_col="Date")
returns.head()

,MASI,CAC40,SP500,EUR_MAD,USD_MAD
Date,,,,,
2015-01-06,0.000891,-0.039694,-0.027014,0.000100,0.009644
2015-01-07,0.010207,0.007158,0.011635,-0.001725,0.002496
2015-01-08,0.005494,0.035855,0.017869,-0.000860,0.003086
2015-01-09,0.019311,-0.019041,-0.008390,0.000421,-0.003724
2015-01-12,0.003362,0.011766,-0.008069,-0.001290,-0.000563


## Descriptive Statistics and Normality

Financial returns often show skewness, excess kurtosis, and Jarque-Bera rejection of normality.

In [3]:
desc = descriptive_statistics(returns)
normality = normality_tests(returns)
corr = correlation_matrix(returns)

desc.to_csv(RESULTS_DIR / "descriptive_statistics_returns.csv")
normality.to_csv(RESULTS_DIR / "jarque_bera_normality_returns.csv")
corr.to_csv(RESULTS_DIR / "correlation_matrix_returns.csv")

desc

,count,mean,median,standard_deviation,min,max,skewness,kurtosis,jarque_bera_statistic,jarque_bera_p_value
variable,,,,,,,,,,
MASI,2729.0,0.000268,0.000299,0.007966,-0.088184,0.054486,-0.854177,14.739918,24938.600536,0.0
CAC40,2729.0,0.000325,0.000724,0.012041,-0.122768,0.083895,-0.799307,10.646342,13126.024461,0.0
SP500,2729.0,0.000541,0.000696,0.011594,-0.119845,0.095154,-0.452872,14.965760,25460.231437,0.0
EUR_MAD,2729.0,-0.000005,-0.000064,0.002851,-0.020584,0.017390,0.159296,3.978405,1802.701311,0.0
USD_MAD,2729.0,0.000014,-0.000049,0.003602,-0.026592,0.018460,-0.097530,3.634732,1499.244412,0.0


In [4]:
normality

,jarque_bera_statistic,p_value,conclusion
variable,,,
MASI,24938.600536,0.0,Reject normality
CAC40,13126.024461,0.0,Reject normality
SP500,25460.231437,0.0,Reject normality
EUR_MAD,1802.701311,0.0,Reject normality
USD_MAD,1499.244412,0.0,Reject normality


## Correlation Matrix

Correlation is not causality, but it provides a first view of market co-movement and potential transmission channels.

In [5]:
plot_correlation_heatmap(returns, FIGURES_DIR / "correlation_heatmap_returns.png")
corr

,MASI,CAC40,SP500,EUR_MAD,USD_MAD
MASI,1.000000,0.213677,0.148777,-0.026814,-0.014793
CAC40,0.213677,1.000000,0.546487,-0.004374,-0.001676
SP500,0.148777,0.546487,1.000000,0.064172,-0.065107
EUR_MAD,-0.026814,-0.004374,0.064172,1.000000,-0.277716
USD_MAD,-0.014793,-0.001676,-0.065107,-0.277716,1.000000


## Outlier Detection

IQR and z-score rules identify extreme observations. These points matter because classical VAR is estimated equation-by-equation with OLS.

In [6]:
iqr_outliers = detect_outliers_iqr(returns)
zscore_outliers = detect_outliers_zscore(returns, threshold=3.0)
outliers = outlier_summary(returns)

iqr_outliers.to_csv(RESULTS_DIR / "iqr_outlier_flags_returns.csv")
zscore_outliers.to_csv(RESULTS_DIR / "zscore_outlier_flags_returns.csv")
outliers.to_csv(RESULTS_DIR / "outlier_summary_returns.csv")

outliers

,iqr_count,iqr_percentage,zscore_count,zscore_percentage
variable,,,,
MASI,191,6.998901,46,1.685599
CAC40,144,5.276658,42,1.539025
SP500,176,6.449249,36,1.319165
EUR_MAD,90,3.297911,42,1.539025
USD_MAD,94,3.444485,38,1.392451


In [7]:
plot_boxplots(returns, FIGURES_DIR / "boxplots_returns.png")
plot_outlier_counts(outliers, FIGURES_DIR / "outlier_counts_returns.png")

## Financial Interpretation

If the Jarque-Bera test rejects normality and outlier counts are material, the data are consistent with common financial stylized facts: fat tails, episodic shocks, and asymmetric movements. This motivates careful interpretation of VAR coefficients and supports a robustness discussion around methods such as VAR-Gini.